# 5차시 (4) 멀티 에이전트 맛보기 & NotebookLM — 실습

한경국립대 2026 여름특강 · 딥러닝/머신러닝 입문 (초급반)

앞에서 만든 챗봇/RAG는 "질문 하나 → 답 하나" 구조였습니다.
이번에는 한 걸음 더 나아가, **여러 역할이 협력하거나 도구를 쓰는** 방식을 가볍게 체험합니다.

- **Part 1. 의도 분류 라우팅** — 들어온 질문의 *의도* 를 먼저 파악해 **알맞은 처리로 보내기**
- **Part 2. 도구 사용(tool use)** — 모델이 스스로 **검색·계산 같은 도구를 호출**해서 답하기
- **Part 3. (개념) 멀티 에이전트 & LangGraph** — 여러 에이전트를 엮는 큰 그림
- **Part 4. NotebookLM** — 코드 없이 *내 문서로 답·요약·오디오/비디오* 를 만드는 도구 사용법

> **에이전트(agent)** = 스스로 *계획하고, 도구를 쓰고, 반복* 하며 일을 해내는 AI. 2026년 가장 검증된 사례는 **코딩 에이전트**(예: Codex, Claude Code)입니다.

## 0. 환경 세팅
라이브러리 설치 + API 키 입력 (앞 노트북과 동일).

In [ ]:
!pip install -qU langchain-openai langchain-core
print("설치 완료!")

In [ ]:
import os
import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API 키를 입력하세요: ")

print("API 키 설정 완료" if os.environ.get("OPENAI_API_KEY") else "키가 비어 있습니다.")

---
# Part 1 · 의도 분류 라우팅 (Routing)

실제 챗봇 서비스는 들어온 질문을 곧장 LLM에 던지지 않고, **먼저 의도를 분류** 한 뒤 알맞은 곳으로 보냅니다.
예: "환불해줘" → 환불 담당 / "오늘 날씨" → 검색 도구 / "우리 회사 휴가 규정" → RAG.

이것을 **라우팅(routing)** 이라 하고, 의도를 가르는 부분이 **라우터(router)** 입니다.
가장 간단한 방법은 **LLM에게 분류를 시키는 것** 입니다.

먼저 의도를 분류하는 라우터를 만들어 봅시다.

In [ ]:
from langchain_openai import ChatOpenAI

router_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

INTENTS = ["날씨", "계산", "일반대화"]

def classify_intent(message):
    prompt = f"""다음 사용자의 말을 아래 의도 중 하나로만 분류해줘. 다른 말 없이 의도 단어만 출력해.

가능한 의도: {', '.join(INTENTS)}

사용자: {message}
의도:"""
    result = router_llm.invoke(prompt).content.strip()
    # 혹시 모를 군더더기 제거: 정의된 의도 중 매칭되는 것만 채택
    for it in INTENTS:
        if it in result:
            return it
    return "일반대화"

# 테스트
for msg in ["내일 서울 날씨 어때?", "123 곱하기 47은?", "심심한데 농담 하나 해줘"]:
    print(f"'{msg}'  ->  의도: {classify_intent(msg)}")

분류된 의도에 따라 **다른 함수(처리기)** 로 보내면 라우팅이 완성됩니다.
각 의도별 처리를 연결해 봅시다(날씨·계산은 일단 단순 처리, 일반대화는 LLM).

In [ ]:
chat_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

def handle_weather(message):
    # (실제로는 날씨 API를 호출합니다. 여기서는 예시 응답)
    return "[날씨 처리기] 날씨 정보는 기상청 API에 연결하면 됩니다. (예: 내일 서울 맑음, 최고 28도)"

def handle_calc(message):
    return "[계산 처리기] 계산이 필요하군요. Part 2에서 '도구 사용'으로 실제 계산을 해봅니다."

def handle_chat(message):
    return "[일반대화] " + chat_llm.invoke(message).content

def route(message):
    intent = classify_intent(message)
    if intent == "날씨":
        return handle_weather(message)
    elif intent == "계산":
        return handle_calc(message)
    else:
        return handle_chat(message)

print(route("오늘 부산 날씨 알려줘"))
print(route("안녕! 오늘 기분 어때?"))

> **연결 포인트**: 직전 노트북(BERT 분류)에서 *의도 데이터* 로 분류기를 학습했다면, 위 `classify_intent` 를 그 **BERT 분류기로 바꿔** 더 빠르고 저렴하게 라우팅할 수 있습니다(LLM 호출 비용↓).
> 또는 2~4차시의 **임베딩·유사도** 를 써서, 각 의도 예시 문장과의 유사도로 분류하는 **시맨틱 라우팅** 도 가능합니다.

---
# Part 2 · 도구 사용 (Tool Use / 함수 호출)

라우터로 의도를 *우리가* 나눠 줄 수도 있지만, 똑똑한 모델은 **스스로 필요한 도구를 골라** 쓰기도 합니다.
이를 **도구 사용(tool use)** 또는 **함수 호출(function calling)** 이라 합니다.

흐름:
1. 우리가 모델에게 "이런 도구들이 있어"라고 **도구 목록(함수)** 을 알려줍니다.
2. 질문이 들어오면 모델이 **"이 도구를 이런 인자로 써!"** 라고 알려줍니다.
3. 우리가 그 도구(함수)를 실제로 **실행** 하고, 결과를 모델에게 돌려줍니다.
4. 모델이 그 결과로 **최종 답** 을 만듭니다.

먼저 모델이 쓸 **도구(함수)** 두 개를 정의합니다: 계산기와 (가짜)검색.

In [ ]:
from langchain_core.tools import tool

@tool
def calculator(expression: str) -> str:
    """수식을 계산한다. 예: '12 * (3 + 4)'. 사칙연산과 괄호를 지원한다."""
    try:
        # 안전을 위해 숫자와 기본 연산자만 허용
        allowed = set("0123456789+-*/(). ")
        if not set(expression) <= allowed:
            return "허용되지 않은 문자가 있습니다."
        return str(eval(expression))
    except Exception as e:
        return f"계산 오류: {e}"

@tool
def web_search(query: str) -> str:
    """인터넷에서 정보를 검색한다. 최신 정보나 사실 확인이 필요할 때 사용한다."""
    # 실제로는 검색 API(예: Tavily, SerpAPI)를 호출합니다.
    # 이번 실습에서는 키 없이 동작하도록 '가짜 검색 결과'를 돌려줍니다.
    fake_db = {
        "한경국립대": "한경국립대학교는 경기도 안성/평택에 캠퍼스를 둔 국립대학교입니다.",
        "파이썬": "파이썬(Python)은 1991년 귀도 반 로섬이 발표한 프로그래밍 언어입니다.",
    }
    for key, val in fake_db.items():
        if key in query:
            return val
    return "(가짜 검색) 관련 결과를 찾지 못했습니다. 실제 서비스에서는 검색 API를 연결하세요."

print("도구 2개 준비 완료: calculator, web_search")

이제 모델에 도구를 **연결(bind)** 하고, 모델이 도구를 호출하면 실제로 실행해 결과를 돌려주는 **루프** 를 만듭니다.
이 루프가 바로 '에이전트' 의 핵심 동작입니다: *모델이 도구를 부르면 → 실행 → 결과를 다시 모델에게 → 최종 답*.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, ToolMessage

tools = [calculator, web_search]
tools_by_name = {t.name: t for t in tools}

# 모델에게 '이런 도구를 쓸 수 있다'고 알려줌
agent_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(tools)

def run_agent(question):
    messages = [HumanMessage(content=question)]

    # 1) 모델에게 질문 -> 모델이 '도구를 쓰자'고 할 수 있음
    ai_msg = agent_llm.invoke(messages)
    messages.append(ai_msg)

    # 2) 모델이 요청한 도구가 있으면 실제로 실행
    if ai_msg.tool_calls:
        for call in ai_msg.tool_calls:
            tool = tools_by_name[call["name"]]
            result = tool.invoke(call["args"])
            print(f"  [도구 실행] {call['name']}({call['args']}) -> {result}")
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
        # 3) 도구 결과를 모델에게 다시 줘서 최종 답 생성
        final = agent_llm.invoke(messages)
        return final.content
    else:
        # 도구가 필요 없으면 바로 답
        return ai_msg.content

print("질문: 137 곱하기 256은 얼마야?")
print("답:", run_agent("137 곱하기 256은 얼마야?"))

In [ ]:
# 검색 도구가 필요한 질문
print("질문: 한경국립대가 어떤 학교야?")
print("답:", run_agent("한경국립대가 어떤 학교야?"))

**TODO.** 아래에 질문을 바꿔 보세요. 모델이 *어떤 도구를 고르는지*(또는 도구 없이 답하는지) `[도구 실행]` 로그로 확인할 수 있습니다.
- 계산이 필요한 질문, 검색이 필요한 질문, 둘 다 필요 없는 잡담을 각각 시험해 보세요.

In [ ]:
# TODO: 질문을 바꿔 도구 선택을 관찰해 보세요.
my_question = "파이썬은 누가 만들었어?"
print("답:", run_agent(my_question))

> **실제 인터넷 검색을 붙이려면**: 위 `web_search` 의 가짜 부분을 실제 검색 API로 바꾸면 됩니다.
> 대표적으로 **Tavily**(`pip install langchain-tavily`, 무료 키 발급)나 SerpAPI 등을 연결합니다. 키가 필요하므로 이번 수업에서는 가짜 검색으로 흐름만 익혔습니다.

---
# Part 3 · (개념) 멀티 에이전트와 LangGraph

지금까지 본 **라우팅** 과 **도구 사용** 을 여러 개 엮으면 **멀티 에이전트 시스템** 이 됩니다.
예: *질문 의도 분류 → (검색 에이전트 / RAG 에이전트 / 계산 에이전트) → 결과를 정리하는 에이전트*.

이런 흐름을 **그래프(graph)** 로 그려 관리하는 대표 도구가 **LangGraph** 입니다.
- **노드(node)**: 하나의 작업(에이전트/함수). 예: '의도 분류', '검색', '답 생성'.
- **엣지(edge)**: 다음에 어디로 갈지(조건 분기 포함).
- **상태(state)**: 노드들이 공유하는 정보(대화·중간 결과).

오늘 우리가 손으로 만든 `route()` 함수가 바로 작은 LangGraph 한 장입니다.

```
           ┌──────────────┐
  질문 ──▶  │  의도 분류    │
           └──────┬───────┘
        ┌─────────┼─────────┐
        ▼         ▼         ▼
   [날씨 검색] [계산 도구] [RAG/대화]
        └─────────┼─────────┘
                  ▼
             최종 답 정리
```

**언제 무엇을 쓰나(2026 기준, 개념만)**
- 복잡하고 분기·상태가 많은 워크플로 → **LangGraph**
- 빠른 프로토타입, 역할 기반 팀 구성 → **CrewAI**
- RAG 중심 검색 에이전트 → **LlamaIndex**

> 초급 단계에서는 "라우팅 + 도구 사용" 개념만 확실히 잡으면 충분합니다. 프레임워크는 필요해질 때 골라 쓰면 됩니다.

---
# Part 4 · NotebookLM — 코드 없이 '내 문서로 답하는' 도구

우리는 (2)번 노트북에서 RAG 챗봇을 **직접 코드로** 만들었습니다.
같은 일을 **코드 없이** 해 주는 구글의 도구가 **NotebookLM** 입니다. 내가 올린 문서(소스)에만 근거해 요약·답변을 해 주는, 일종의 **개인용 RAG** 입니다.

## NotebookLM 으로 할 수 있는 것 (2026 기준)
올린 문서를 바탕으로 **Studio** 패널에서 다음을 만들 수 있습니다:
- **Audio Overview**: AI 진행자 2명이 내 문서를 *팟캐스트처럼 대화로* 요약(80개 이상 언어 지원).
- **Video Overview**: 문서의 이미지·도표·수치를 끌어와 *내레이션 슬라이드 영상* 으로 설명.
- **Mind Map**: 문서 내용을 *마인드맵* 으로 구조화.
- **Reports**: 요약 리포트·FAQ·학습 가이드 등 문서 생성.
- 채팅으로 "이 자료에서 ○○ 알려줘" 라고 물으면 **출처(인용)** 와 함께 답해 줍니다.

## 직접 해보기 (단계)
1. 브라우저에서 **notebooklm.google.com** 접속 → 구글 계정 로그인.
2. **[새로 만들기] / [Create]** → 소스 추가: PDF, Google Docs, 웹페이지 URL, 붙여넣은 텍스트 등.
3. 소스가 처리되면, 가운데 채팅창에 질문해 봅니다("3쪽 핵심만 요약해줘").
4. 오른쪽 **Studio** 에서 **Audio Overview** 또는 **Mind Map** 을 생성해 봅니다.
5. (선택) 집중 주제·대상 청중을 지정해 결과물을 맞춤화할 수 있습니다.

> 실습 아이디어: (2)번 노트북에서 만든 *사내 안내 문서* 같은 텍스트를 NotebookLM 에 올려, **직접 만든 RAG 챗봇** 과 **NotebookLM** 의 답을 비교해 보세요.

### 화면 캡처 자리 (강사/수강생용)

아래는 NotebookLM 사용 화면을 붙여 넣을 자리입니다. (이 노트북은 코드가 아니라 **직접 캡처** 가 필요합니다.)
수업 중 본인 화면을 캡처해 아래 설명과 함께 참고하세요. 캡처하면 좋은 4장:

1. **소스 업로드 화면** — 문서를 추가하는 화면
2. **채팅 답변 + 인용 표시** — 답변 옆에 출처 번호가 붙은 모습
3. **Studio 패널** — Audio/Video/Mind Map/Reports 타일이 보이는 화면
4. **Audio Overview 생성 결과** — 오디오 요약이 만들어진 모습

> (이미지 자리) — 캡처한 그림을 이 셀 아래에 끼워 넣거나, 별도 문서로 정리하세요.

---
## 마무리

오늘 (4)번 노트북에서 체험한 것:
- **라우팅**: 의도를 먼저 분류해 알맞은 처리로 보내기(LLM 라우터 / BERT·임베딩으로 대체 가능).
- **도구 사용**: 모델이 스스로 계산기·검색 같은 도구를 호출 → 실행 → 결과로 최종 답.
- **멀티 에이전트 & LangGraph**(개념): 라우팅+도구를 여러 개 엮어 협업하는 큰 그림.
- **NotebookLM**: 코드 없이 내 문서로 답·요약·오디오/비디오를 만드는 도구.

**5차시 전체 정리**: 프롬프트 엔지니어링 → 나만의 챗봇 → RAG 챗봇 → BERT 분류 → 멀티 에이전트·NotebookLM.
작은 부품(프롬프트·임베딩·검색·분류·도구)을 조합하면 점점 똑똑한 AI 서비스가 됩니다. 수고하셨습니다!